# MVPA — 02: Statistics and Visualization

Runs cluster-based permutation tests on decoding scores and generates
publication-ready figures.

Covers:
- Real vs null decoding (one analysis)
- Temporal generalization matrix
- Comparing multiple analyses (e.g. target vs distractor)

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt

from eeg_toolkit import load_config
from eeg_toolkit.mvpa import load_all_scores
from eeg_toolkit.mvpa_stats import test_real_vs_null, compare_analyses

# ── Update this path ──
cfg = load_config('../../configs/your_experiment.yaml')

# ── Load decoding scores ──
# analysis_name must match what you used in notebook 01
group = load_all_scores(cfg, 'your_window', 'a_vs_b')
print(f"Scores shape: {group['subject_scores'].shape}")

In [ ]:
# ── Cluster test: real vs null ──
results = test_real_vs_null(group, n_permutations=10000)

# ── Plot ──
times = group['times']
scores = group['subject_scores']
null = group['subject_null_scores']
n_subj = scores.shape[0]

mean_acc = scores.mean(axis=0)
sem_acc  = scores.std(axis=0) / np.sqrt(n_subj)
mean_null = null.mean(axis=1).mean(axis=0)
sem_null  = null.mean(axis=1).std(axis=0) / np.sqrt(n_subj)

fig, (ax, ax_sig) = plt.subplots(2, 1, figsize=(8, 5),
                                  gridspec_kw={'height_ratios': [5, 1]},
                                  sharex=True)

ax.plot(times, mean_acc, color='#2166AC', lw=2.5, label='Real')
ax.fill_between(times, mean_acc - sem_acc, mean_acc + sem_acc,
                color='#2166AC', alpha=0.15)
ax.plot(times, mean_null, color='gray', lw=1.8, ls='--', label='Null')
ax.fill_between(times, mean_null - sem_null, mean_null + sem_null,
                color='gray', alpha=0.1)
ax.axhline(0.5, color='k', lw=0.8, ls=':')
ax.axvline(0, color='k', lw=1.5)
ax.set_ylabel('Accuracy')
ax.legend()

for cluster in results.get('significant_clusters', []):
    t0, t1 = times[cluster['start_idx']], times[cluster['end_idx']]
    ax_sig.barh(0, t1 - t0, left=t0, height=0.6,
                color='#2166AC', alpha=0.85)
ax_sig.axvline(0, color='k', lw=1.5)
ax_sig.set_yticks([0])
ax_sig.set_yticklabels(['Real > Null'], fontsize=8)
ax_sig.set_xlabel('Time (s)')

plt.tight_layout()
plt.show()

In [ ]:
# ── Optional: temporal generalization matrix ──
from scipy.stats import ttest_1samp
from mne.stats import fdr_correction

group_tg = load_all_scores(cfg, 'your_window', 'a_vs_b_tg')
tg = group_tg['subject_temporal_gen']    # (n_subjects, n_times, n_times)
times_tg = group_tg['times']

mean_tg = tg.mean(axis=0)
t_vals, p_vals = ttest_1samp(tg, 0.5, axis=0)
reject, _ = fdr_correction(p_vals, alpha=0.05)

vdev = max(abs(mean_tg.max() - 0.5), abs(mean_tg.min() - 0.5))
extent = [times_tg[0], times_tg[-1], times_tg[0], times_tg[-1]]

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(mean_tg, origin='lower', extent=extent,
          cmap='RdBu_r', vmin=0.5-vdev, vmax=0.5+vdev,
          aspect='equal', alpha=0.3)
mean_tg_masked = np.ma.masked_where(~reject, mean_tg)
im = ax.imshow(mean_tg_masked, origin='lower', extent=extent,
               cmap='RdBu_r', vmin=0.5-vdev, vmax=0.5+vdev,
               aspect='equal', alpha=1.0)
ax.axhline(0, color='k', lw=0.5, ls=':')
ax.axvline(0, color='k', lw=0.5, ls=':')
ax.set_xlabel('Test time (s)')
ax.set_ylabel('Train time (s)')
fig.colorbar(im, ax=ax, label='Accuracy')
plt.tight_layout()
plt.show()

n_sig = reject.sum()
print(f"FDR significant: {n_sig}/{reject.size} ({100*n_sig/reject.size:.1f}%)")

In [ ]:
# ── Optional: compare two analyses ──
# Load a second analysis and compare it against the first.
group_b = load_all_scores(cfg, 'your_window', 'another_analysis')

results_a = test_real_vs_null(group,   n_permutations=10000)
results_b = test_real_vs_null(group_b, n_permutations=10000)
results_comp = compare_analyses(group, group_b, n_permutations=10000)